# TIR Experiment Results

Plots and tables for CS224R final project.

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 150

EVAL_DIR = '/Users/sbfisher/Stanford/CS224R/final_project/eval_results'

IndentationError: unexpected indent (3193640399.py, line 11)

## 1. SFT Ablation Results

In [5]:
def pass_at_k(n, c, k):
    """Unbiased pass@k estimator."""
    if n - c < k:
        return 1.0
    return 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1))

def compute_pass_at_k(scores_list, k_values=[1, 4, 16]):
    """Compute unbiased pass@k from a list of per-problem score lists."""
    results = {}
    for k in k_values:
        vals = []
        for scores in scores_list:
            n = len(scores)
            c = sum(1 for s in scores if s >= 1.0)
            if k <= n:
                vals.append(pass_at_k(n, c, k))
        results[f'pass@{k}'] = np.mean(vals) if vals else 0.0
    return results

def load_eval_results(eval_dir):
    """Load all JSONL eval result files."""
    rows = []
    for f in sorted(os.listdir(eval_dir)):
        if not f.endswith('.json'):
            continue
        path = os.path.join(eval_dir, f)
        with open(path) as fh:
            lines = [json.loads(l) for l in fh if l.strip()]
        if not lines or 'scores' not in lines[0]:
            continue
        scores_list = [l['scores'] for l in lines]
        metrics = compute_pass_at_k(scores_list)
        metrics['name'] = f.replace('.json', '')
        metrics['n_problems'] = len(lines)
        metrics['n_samples'] = len(lines[0]['scores'])
        rows.append(metrics)
    return pd.DataFrame(rows)

all_results = load_eval_results(EVAL_DIR)
all_results

FileNotFoundError: [Errno 2] No such file or directory: 'eval_results'

In [ ]:
# SFT ablation subset — v2 results only
sft_names = [
    '3tool_from_sft_v2_with_tools', '3tool_from_sft_v2_no_tools',
    'calc_only_from_sft_v2_with_tools', 'calc_only_from_sft_v2_no_tools',
    '3tool_from_base_v2_with_tools', '3tool_from_base_v2_no_tools',
    'calc_only_from_base_v2_with_tools', 'calc_only_from_base_v2_no_tools',
    'sft_eval_run', 'downloaded_eval_results',
]
sft_df = all_results[all_results['name'].isin(sft_names)].copy()

# Parse model attributes
def parse_sft_name(name):
    if name in ('sft_eval_run', 'downloaded_eval_results'):
        return {'tools_trained': 'none', 'warm_start': 'N/A',
                'eval_tools': 'no', 'label': 'Vanilla SFT'}
    tools_trained = '3tool' if '3tool' in name else 'calc_only'
    warm_start = 'from_sft' if 'from_sft' in name else 'from_base'
    eval_tools = 'with' if 'with_tools' in name else 'no'
    label = f"{tools_trained} {warm_start}"
    return {'tools_trained': tools_trained, 'warm_start': warm_start,
            'eval_tools': eval_tools, 'label': label}

attrs = pd.DataFrame([parse_sft_name(n) for n in sft_df['name']])
sft_df = pd.concat([sft_df.reset_index(drop=True), attrs], axis=1)
sft_df[['label', 'eval_tools', 'pass@1', 'pass@4', 'pass@16']].sort_values('pass@1', ascending=False)

In [ ]:
# Pivot table: model vs eval mode
sft_pivot = sft_df.pivot_table(
    index='label', columns='eval_tools', values='pass@1'
).rename(columns={'with': 'With Tools', 'no': 'No Tools'})
sft_pivot = sft_pivot.sort_values('With Tools', ascending=False)
print("SFT Ablation — pass@1 (unbiased estimator, n=50, k=16)")
print("=" * 55)
display(sft_pivot.style.format('{:.1%}').set_caption('pass@1 by Model and Eval Mode'))

In [ ]:
# Overlapping bar chart: pass@1, pass@4, pass@16 for With Tools vs No Tools
# Wider bars behind narrower ones so all three k values are visible

k_metrics = ['pass@16', 'pass@4', 'pass@1']  # draw widest (highest) first
bar_widths = [0.35, 0.25, 0.15]
alphas = [0.45, 0.65, 1.0]

# Pivot for each k
pivots = {}
for k in ['pass@1', 'pass@4', 'pass@16']:
    piv = sft_df.pivot_table(index='label', columns='eval_tools', values=k)
    piv = piv.rename(columns={'with': 'With Tools', 'no': 'No Tools'})
    pivots[k] = piv

# Use pass@1 With Tools ordering
order = pivots['pass@1'].sort_values('With Tools', ascending=False).index
for k in pivots:
    pivots[k] = pivots[k].reindex(order)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(order))
half = 0.22  # offset between With/No groups

colors_with = '#2196F3'
colors_no = '#FF9800'

for metric, w, alpha in zip(k_metrics, bar_widths, alphas):
    piv = pivots[metric]
    ax.bar(x - half, piv['With Tools'] * 100, w, color=colors_with, alpha=alpha, edgecolor='white', linewidth=0.5)
    if 'No Tools' in piv.columns:
        vals = piv['No Tools'].fillna(0) * 100
        ax.bar(x + half, vals, w, color=colors_no, alpha=alpha, edgecolor='white', linewidth=0.5)

# Value labels for pass@1 only (smallest/front bars)
piv1 = pivots['pass@1']
for i, lbl in enumerate(order):
    v = piv1.loc[lbl, 'With Tools']
    if not np.isnan(v):
        ax.text(i - half, v * 100 + 1, f'{v*100:.1f}', ha='center', va='bottom', fontsize=7)
    if 'No Tools' in piv1.columns:
        v = piv1.loc[lbl, 'No Tools']
        if not np.isnan(v):
            ax.text(i + half, v * 100 + 1, f'{v*100:.1f}', ha='center', va='bottom', fontsize=7)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=colors_with, alpha=1.0, label='With Tools'),
    Patch(facecolor=colors_no, alpha=1.0, label='No Tools'),
    Patch(facecolor='gray', alpha=0.45, label='pass@16'),
    Patch(facecolor='gray', alpha=0.65, label='pass@4'),
    Patch(facecolor='gray', alpha=1.0, label='pass@1'),
]
ax.legend(handles=legend_elements, frameon=False, fontsize=10, ncol=2)

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('SFT Ablation: Tool Access at Evaluation (pass@1/4/16)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(order, rotation=35, ha='right', fontsize=9)
ax.set_ylim(0, 100)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(f'{EVAL_DIR}/sft_ablation_bar.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# Line plot: pass@k vs k for each SFT ablation
k_values = [1, 4, 16]

# Build lines: one per (label, eval_tools) combo
fig, ax = plt.subplots(figsize=(10, 6))

# Distinct styles for with/no tools
style_map = {'with': '-o', 'no': '--s'}
cmap = plt.cm.tab10

# Get unique labels ordered by pass@1 with tools
labels_ordered = sft_df[sft_df['eval_tools'] == 'with'].sort_values('pass@1', ascending=False)['label'].tolist()
# Add labels that only appear in no-tools (e.g. Vanilla SFT)
for lbl in sft_df['label'].unique():
    if lbl not in labels_ordered:
        labels_ordered.append(lbl)

for i, label in enumerate(labels_ordered):
    color = cmap(i)
    for eval_mode in ['with', 'no']:
        row = sft_df[(sft_df['label'] == label) & (sft_df['eval_tools'] == eval_mode)]
        if row.empty:
            continue
        row = row.iloc[0]
        vals = [row['pass@1'] * 100, row['pass@4'] * 100, row['pass@16'] * 100]
        suffix = ' (tools)' if eval_mode == 'with' else ' (no tools)'
        ax.plot(k_values, vals, style_map[eval_mode], color=color,
                label=f'{label}{suffix}', markersize=6, linewidth=1.5)

ax.set_xlabel('k', fontsize=12)
ax.set_ylabel('pass@k (%)', fontsize=12)
ax.set_title('SFT Ablation: pass@k vs k', fontsize=14)
ax.set_xticks(k_values)
ax.set_xticklabels([str(k) for k in k_values])
ax.set_ylim(0, 100)
ax.legend(fontsize=8, ncol=2, frameon=False, loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(f'{EVAL_DIR}/sft_passk_lines.png', bbox_inches='tight', dpi=300)
plt.show()

## 2. RLOO Evaluation Results

In [ ]:
# RLOO evaluation results — latest checkpoints only
rloo_names = [
    'rloo_vanilla_latest_with_tools', 'rloo_vanilla_latest_no_tools',
    'rloo_nosc_latest_with_tools', 'rloo_nosc_latest_no_tools',
    'rloo_sc_step50_with_tools', 'rloo_sc_step50_no_tools',
]
rloo_df = all_results[all_results['name'].isin(rloo_names)].copy()

def parse_rloo_name(name):
    eval_tools = 'with' if 'with_tools' in name else 'no'
    base = name.replace('_with_tools', '').replace('_no_tools', '')
    label_map = {
        'rloo_vanilla_latest': 'Vanilla',
        'rloo_nosc_latest': 'Hierarchical',
        'rloo_sc_step50': 'Self-Critic (step 50)',
    }
    label = label_map.get(base, base)
    return {'eval_tools': eval_tools, 'label': label}

attrs = pd.DataFrame([parse_rloo_name(n) for n in rloo_df['name']])
rloo_df = pd.concat([rloo_df.reset_index(drop=True), attrs], axis=1)
rloo_df[['label', 'eval_tools', 'pass@1', 'pass@4', 'pass@16']].sort_values('pass@1', ascending=False)

In [ ]:
# Line plot: pass@k vs k for each RLOO model
k_values = [1, 4, 16]

fig, ax = plt.subplots(figsize=(10, 6))

style_map = {'with': '-o', 'no': '--s'}
cmap = plt.cm.tab10

labels_ordered = rloo_df[rloo_df['eval_tools'] == 'with'].sort_values('pass@1', ascending=False)['label'].tolist()
for lbl in rloo_df['label'].unique():
    if lbl not in labels_ordered:
        labels_ordered.append(lbl)

for i, label in enumerate(labels_ordered):
    color = cmap(i)
    for eval_mode in ['with', 'no']:
        row = rloo_df[(rloo_df['label'] == label) & (rloo_df['eval_tools'] == eval_mode)]
        if row.empty:
            continue
        row = row.iloc[0]
        vals = [row['pass@1'] * 100, row['pass@4'] * 100, row['pass@16'] * 100]
        suffix = ' (tools)' if eval_mode == 'with' else ' (no tools)'
        ax.plot(k_values, vals, style_map[eval_mode], color=color,
                label=f'{label}{suffix}', markersize=6, linewidth=1.5)

ax.set_xlabel('k', fontsize=12)
ax.set_ylabel('pass@k (%)', fontsize=12)
ax.set_title('RLOO Evaluation: pass@k vs k', fontsize=14)
ax.set_xticks(k_values)
ax.set_xticklabels([str(k) for k in k_values])
ax.set_ylim(0, 100)
ax.legend(fontsize=8, ncol=2, frameon=False, loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(f'{EVAL_DIR}/rloo_passk_lines.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# RLOO bar chart: pass@1/4/16, With Tools vs No Tools (overlapping bars)
from matplotlib.patches import Patch

k_metrics = ['pass@16', 'pass@4', 'pass@1']
bar_widths = [0.35, 0.25, 0.15]
alphas = [0.45, 0.65, 1.0]

pivots_rloo = {}
for k in ['pass@1', 'pass@4', 'pass@16']:
    piv = rloo_df.pivot_table(index='label', columns='eval_tools', values=k)
    piv = piv.rename(columns={'with': 'With Tools', 'no': 'No Tools'})
    pivots_rloo[k] = piv

# Order by pass@1 With Tools (descending)
has_with = 'With Tools' in pivots_rloo['pass@1'].columns
sort_col = 'With Tools' if has_with else 'No Tools'
order = pivots_rloo['pass@1'].sort_values(sort_col, ascending=False, na_position='last').index
for k in pivots_rloo:
    pivots_rloo[k] = pivots_rloo[k].reindex(order)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(order))
half = 0.22

colors_with = '#2196F3'
colors_no = '#FF9800'

for metric, w, alpha in zip(k_metrics, bar_widths, alphas):
    piv = pivots_rloo[metric]
    if 'With Tools' in piv.columns:
        ax.bar(x - half, piv['With Tools'].fillna(0) * 100, w, color=colors_with, alpha=alpha, edgecolor='white', linewidth=0.5)
    if 'No Tools' in piv.columns:
        ax.bar(x + half, piv['No Tools'].fillna(0) * 100, w, color=colors_no, alpha=alpha, edgecolor='white', linewidth=0.5)

# Value labels for pass@1
piv1 = pivots_rloo['pass@1']
for i, lbl in enumerate(order):
    if 'With Tools' in piv1.columns and not pd.isna(piv1.loc[lbl, 'With Tools']):
        v = piv1.loc[lbl, 'With Tools']
        ax.text(i - half, v * 100 + 1, f'{v*100:.1f}', ha='center', va='bottom', fontsize=7)
    if 'No Tools' in piv1.columns and not pd.isna(piv1.loc[lbl, 'No Tools']):
        v = piv1.loc[lbl, 'No Tools']
        ax.text(i + half, v * 100 + 1, f'{v*100:.1f}', ha='center', va='bottom', fontsize=7)

legend_elements = [
    Patch(facecolor=colors_with, alpha=1.0, label='With Tools'),
    Patch(facecolor=colors_no, alpha=1.0, label='No Tools'),
    Patch(facecolor='gray', alpha=0.45, label='pass@16'),
    Patch(facecolor='gray', alpha=0.65, label='pass@4'),
    Patch(facecolor='gray', alpha=1.0, label='pass@1'),
]
ax.legend(handles=legend_elements, frameon=False, fontsize=10, ncol=2)

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('RLOO Evaluation: Tool Access at Evaluation (pass@1/4/16)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(order, rotation=35, ha='right', fontsize=9)
ax.set_ylim(0, 100)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(f'{EVAL_DIR}/rloo_eval_bar.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
import re

def has_tool_call(response):
    """Check if a response contains tool calls (either <tool> or <use_tool> format)."""
    return bool(re.search(r'<tool>|<use_tool>', response))

def count_tool_calls(response):
    """Count number of tool invocations in a response."""
    return len(re.findall(r'<tool>|<use_tool>', response))

def load_tool_stats(eval_dir, filenames):
    """Load per-response tool usage stats from with-tools eval files."""
    rows = []
    for fname in filenames:
        path = os.path.join(eval_dir, fname + '.json')
        if not os.path.exists(path):
            continue
        with open(path) as f:
            problems = [json.loads(l) for l in f if l.strip()]
        for pid, prob in enumerate(problems):
            for rid, (resp, score) in enumerate(zip(prob['response'], prob['scores'])):
                used_tool = has_tool_call(resp)
                n_tools = count_tool_calls(resp)
                correct = score >= 1.0
                rows.append({
                    'file': fname, 'problem_id': pid, 'response_id': rid,
                    'used_tool': used_tool, 'n_tool_calls': n_tools,
                    'correct': correct, 'score': score,
                })
    return pd.DataFrame(rows)

# Load with-tools evals
with_tools_files = [
    'rloo_vanilla_latest_with_tools',
    'rloo_nosc_latest_with_tools',
    'rloo_sc_step50_with_tools',
]
label_map = {
    'rloo_vanilla_latest_with_tools': 'Vanilla',
    'rloo_nosc_latest_with_tools': 'Hierarchical',
    'rloo_sc_step50_with_tools': 'Self-Critic (step 50)',
}

tool_df = load_tool_stats(EVAL_DIR, with_tools_files)
tool_df['label'] = tool_df['file'].map(label_map)

# Summary table
summary = tool_df.groupby('label').agg(
    total_responses=('used_tool', 'count'),
    pct_used_tools=('used_tool', 'mean'),
    avg_tool_calls=('n_tool_calls', 'mean'),
    accuracy_all=('correct', 'mean'),
).copy()

# Accuracy split by tool use
for label in summary.index:
    sub = tool_df[tool_df['label'] == label]
    with_t = sub[sub['used_tool']]
    without_t = sub[~sub['used_tool']]
    summary.loc[label, 'accuracy_with_tools'] = with_t['correct'].mean() if len(with_t) > 0 else np.nan
    summary.loc[label, 'accuracy_without_tools'] = without_t['correct'].mean() if len(without_t) > 0 else np.nan
    summary.loc[label, 'pct_no_tools'] = len(without_t) / len(sub)

summary = summary.rename(columns={
    'pct_used_tools': '% Used Tools',
    'avg_tool_calls': 'Avg Tool Calls',
    'accuracy_all': 'Overall Acc',
    'accuracy_with_tools': 'Acc (used tools)',
    'accuracy_without_tools': 'Acc (no tools)',
})

print("Tool Usage Summary (with-tools eval mode)")
print("=" * 60)
display(summary[['% Used Tools', 'Avg Tool Calls', 'Overall Acc',
                  'Acc (used tools)', 'Acc (no tools)']].style.format({
    '% Used Tools': '{:.1%}', 'Avg Tool Calls': '{:.2f}',
    'Overall Acc': '{:.1%}', 'Acc (used tools)': '{:.1%}', 'Acc (no tools)': '{:.1%}',
}))

In [ ]:
# Tool usage bar chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

labels_ordered = ['Vanilla', 'Hierarchical', 'Self-Critic (step 50)']

# 1. % responses using tools
ax = axes[0]
vals = [summary.loc[l, '% Used Tools'] * 100 if l in summary.index else 0 for l in labels_ordered]
bars = ax.bar(labels_ordered, vals, color=['#2196F3', '#4CAF50', '#FF5722'], edgecolor='white')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 1, f'{v:.0f}%', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('% Responses', fontsize=11)
ax.set_title('Tool Usage Rate', fontsize=13)
ax.set_ylim(0, 110)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# 2. Avg tool calls per response
ax = axes[1]
vals = [summary.loc[l, 'Avg Tool Calls'] if l in summary.index else 0 for l in labels_ordered]
bars = ax.bar(labels_ordered, vals, color=['#2196F3', '#4CAF50', '#FF5722'], edgecolor='white')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.05, f'{v:.1f}', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Avg Tool Calls', fontsize=11)
ax.set_title('Tool Calls per Response', fontsize=13)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# 3. Accuracy: used tools vs didn't
ax = axes[2]
x = np.arange(len(labels_ordered))
w = 0.35
acc_with = [summary.loc[l, 'Acc (used tools)'] * 100 if l in summary.index and not pd.isna(summary.loc[l, 'Acc (used tools)']) else 0 for l in labels_ordered]
acc_without = [summary.loc[l, 'Acc (no tools)'] * 100 if l in summary.index and not pd.isna(summary.loc[l, 'Acc (no tools)']) else 0 for l in labels_ordered]
b1 = ax.bar(x - w/2, acc_with, w, label='Used Tools', color='#2196F3', alpha=0.8, edgecolor='white')
b2 = ax.bar(x + w/2, acc_without, w, label='No Tools', color='#FF9800', alpha=0.8, edgecolor='white')
for bar in b1:
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{bar.get_height():.0f}%', ha='center', va='bottom', fontsize=8)
for bar in b2:
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{bar.get_height():.0f}%', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(labels_ordered, fontsize=9)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('Accuracy by Tool Use', fontsize=13)
ax.set_ylim(0, 100)
ax.legend(frameon=False, fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(f'{EVAL_DIR}/rloo_tool_usage.png', bbox_inches='tight', dpi=300)
plt.show()

## 3. Tool Usage Analysis (RLOO with-tools evals)

In [ ]:
# Per-problem crossover: compare with-tools vs no-tools eval on same problems
# For each model, classify each problem as:
#   - Both correct: solved with AND without tools
#   - Tools helped: solved with tools, failed without
#   - Tools hurt: failed with tools, solved without
#   - Both failed: failed in both modes

pairs = [
    ('rloo_vanilla_latest_with_tools', 'rloo_vanilla_latest_no_tools', 'Vanilla'),
    ('rloo_nosc_latest_with_tools', 'rloo_nosc_latest_no_tools', 'Hierarchical'),
    ('rloo_sc_step50_with_tools', 'rloo_sc_step50_no_tools', 'Self-Critic (step 50)'),
]

crossover_rows = []
for wf, nf, label in pairs:
    wp = os.path.join(EVAL_DIR, wf + '.json')
    np_ = os.path.join(EVAL_DIR, nf + '.json')
    if not os.path.exists(wp) or not os.path.exists(np_):
        continue
    with open(wp) as f:
        w_probs = [json.loads(l) for l in f if l.strip()]
    with open(np_) as f:
        n_probs = [json.loads(l) for l in f if l.strip()]

    both_correct = tools_helped = tools_hurt = both_failed = 0
    for w, n in zip(w_probs, n_probs):
        # Use majority vote (>50% of samples correct)
        w_correct = np.mean([s >= 1.0 for s in w['scores']]) > 0.5
        n_correct = np.mean([s >= 1.0 for s in n['scores']]) > 0.5
        if w_correct and n_correct:
            both_correct += 1
        elif w_correct and not n_correct:
            tools_helped += 1
        elif not w_correct and n_correct:
            tools_hurt += 1
        else:
            both_failed += 1
    total = len(w_probs)
    crossover_rows.append({
        'Model': label,
        'Both Correct': both_correct, 'Tools Helped': tools_helped,
        'Tools Hurt': tools_hurt, 'Both Failed': both_failed,
        'Total': total,
    })

cross_df = pd.DataFrame(crossover_rows).set_index('Model')

# Stacked bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(cross_df))
w = 0.5
colors = ['#4CAF50', '#2196F3', '#FF5722', '#9E9E9E']
labels_cat = ['Both Correct', 'Tools Helped', 'Tools Hurt', 'Both Failed']

bottom = np.zeros(len(cross_df))
for cat, color in zip(labels_cat, colors):
    vals = cross_df[cat].values / cross_df['Total'].values * 100
    ax.bar(x, vals, w, bottom=bottom, label=cat, color=color, edgecolor='white', linewidth=0.5)
    # Add count labels
    for i, (v, cnt) in enumerate(zip(vals, cross_df[cat].values)):
        if v > 3:
            ax.text(i, bottom[i] + v/2, f'{cnt}', ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels(cross_df.index, fontsize=11)
ax.set_ylabel('% of Problems', fontsize=12)
ax.set_title('Per-Problem Crossover: With Tools vs No Tools (majority vote)', fontsize=13)
ax.set_ylim(0, 105)
ax.legend(frameon=False, fontsize=10, loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(f'{EVAL_DIR}/rloo_crossover.png', bbox_inches='tight', dpi=300)
plt.show()

# Also show as table
pct_df = cross_df[labels_cat].div(cross_df['Total'], axis=0)
display(pct_df.style.format('{:.0%}').set_caption('Per-problem crossover (majority vote)'))